In [57]:
!pip install groq

In [58]:
from groq import Groq
from google.colab import userdata
import json
import csv

client_groq = Groq(api_key=userdata.get('GROQ_API_KEY'))
client = Groq(api_key=userdata.get('GROQ_API_KEY'))


In [59]:
def analyse_customer_structured(client, customer):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a banking risk analyst.
                Always respond with ONLY valid JSON, no other text.
                Use exactly this structure:
                {
                    "customer_name": "string",
                    "risk_score": number between 1-10,
                    "risk_level": "Low" or "Medium" or "High",
                    "reason": "one sentence explanation",
                    "recommendation": "one sentence action"
                }"""
            },
            {
                "role": "user",
                "content": f"""Analyse this customer:
                Name: {customer['name']}
                Balance: € {customer['balance']}
                Transactions: {customer['transactions']}
                Active: {customer['is_active']}"""
            }
        ]
    )
    return json.loads(response.choices[0].message.content)




In [60]:
from groq import Groq
import json
from google.colab import userdata



def generate_user_story(requirement):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior AI Product Owner with
                10 years experience in banking and financial services.

                When given a business requirement, respond with ONLY
                valid JSON in exactly this structure:
                {
                    "user_story": "As a [role] I want [feature] so that [benefit]",
                    "acceptance_criteria": [
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]",
                        "Given [context] When [action] Then [outcome]"
                    ],
                    "definition_of_done": [
                        "criterion 1",
                        "criterion 2",
                        "criterion 3"
                    ],
                    "story_points": number,
                    "priority": "High" or "Medium" or "Low"
                }"""
            },
            {
                "role": "user",
                "content": f"Generate a user story for this requirement: {requirement}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [61]:
from groq import Groq
from google.colab import userdata
import json


def check_eu_ai_act(use_case):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are an EU AI Act compliance expert
                specialising in financial services.

                Analyse AI use cases and classify them under the EU AI Act.

                Respond with ONLY valid JSON in exactly this structure:
                {
                    "use_case_summary": "brief summary of the use case",
                    "risk_tier": "Unacceptable" or "High Risk" or "Limited Risk" or "Minimal Risk",
                    "risk_tier_reason": "why this tier applies",
                    "compliance_requirements": [
                        "requirement 1",
                        "requirement 2",
                        "requirement 3"
                    ],
                    "red_flags": [
                        "red flag 1",
                        "red flag 2"
                    ],
                    "recommended_actions": [
                        "action 1",
                        "action 2",
                        "action 3"
                    ],
                    "compliant_by": "date by which compliance is required"
                }"""
            },
            {
                "role": "user",
                "content": f"Analyse this AI use case for EU AI Act compliance: {use_case}"
            }
        ]
    )
    return json.loads(response.choices[0].message.content)



In [62]:
def assess_taxonomy(client):

    if not client['nace_code']:
        return {
            "client_id": client['client_id'],
            "client_name": client['client_name'],
            "assessment_tier": "CANNOT ASSESS",
            "taxonomy_eligible": None,
            "taxonomy_aligned": None,
            "substantial_contribution": None,
            "dnsh_assessment": None,
            "minimum_safeguards": None,
            "revenue_kpi": None,
            "capex_kpi": None,
            "gaps": ["Insufficient data for any assessment"],
            "recommendations": ["Collect NACE code, sector, GHG data and safeguards confirmation from client"]
        }

    # Build context
    capex_display = f"€ {client['capex_eur']:,}" if client['capex_eur'] else 'Not provided'

    context = f"""
    Client: {client['client_name']}
    Sector: {client['sector'] or 'Unknown'}
    NACE Code: {client['nace_code']}
    Main Purpose: {client['main_purpose'] or 'Not provided'}
    Sub Purpose: {client['sub_purpose'] or 'Not provided'}
    Product Type: {client['product_type']}
    Reporting Year: {client['reporting_year']}
    Substantial Contribution Objective: {client['substantial_contribution_objective'] or 'Not provided'}
    Revenue Eligible %: {client['revenue_eligible_pct'] or 'Not provided'}
    CapEx Eligible %: {client['capex_eligible_pct'] or 'Not provided'}
    GHG Scope 1+2 (tonnes): {client['ghg_scope1_2_tonnes'] or 'Not provided'}
    Minimum Safeguards Compliant: {client['minimum_safeguards_compliant']}
    Annual Turnover: € {client['annual_turnover_eur']:,}
    CapEx: {capex_display}
    """

    if client['nace_code'] and client['main_purpose'] and client['sub_purpose'] and client['ghg_scope1_2_tonnes']:
        instruction = "Perform a complete 4-step EU Taxonomy assessment."
    elif client['nace_code'] and client['main_purpose']:
        instruction = "Perform a partial EU Taxonomy assessment. Check eligibility and substantial contribution only."
    else:
        instruction = "Perform an eligibility check only based on NACE code."

    response = client_groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "system",
                "content": """You are a senior EU Taxonomy compliance expert at a Dutch bank.
You follow the official EU Taxonomy Compass 4-step framework strictly.

CRITICAL RULES:
- Diesel or fossil fuel transport is NOT taxonomy aligned
- Traditional blast furnace steel is NOT taxonomy aligned
- Only activities meeting Technical Screening Criteria are aligned
- Be conservative — when in doubt mark as not aligned

Respond with ONLY valid JSON. No preamble, no markdown.

                Use exactly this structure:
                {
                    "assessment_tier": "FULL" or "PARTIAL" or "ELIGIBILITY",
                    "taxonomy_eligible": true or false,
                    "taxonomy_aligned": true or false or null,
                    "substantial_contribution": {
                        "objective": "objective name",
                        "meets_criteria": true or false,
                        "reasoning": "one sentence"
                    },
                    "dnsh_assessment": {
                        "climate_mitigation": "Pass" or "Fail" or "Incomplete",
                        "climate_adaptation": "Pass" or "Fail" or "Incomplete",
                        "water": "Pass" or "Fail" or "Incomplete",
                        "circular_economy": "Pass" or "Fail" or "Incomplete",
                        "pollution": "Pass" or "Fail" or "Incomplete",
                        "biodiversity": "Pass" or "Fail" or "Incomplete",
                        "overall": "Pass" or "Fail" or "Incomplete"
                    },
                    "minimum_safeguards": "Pass" or "Fail" or "Incomplete",
                    "revenue_kpi": number or null,
                    "capex_kpi": number or null,
                    "gaps": ["gap 1", "gap 2"],
                    "recommendations": ["action 1", "action 2"]
                }"""
            },
            {
                "role": "user",
                "content": f"{instruction}\n\nClient data:\n{context}"
            }
        ]
    )

    # Debug — see raw response
    raw = response.choices[0].message.content
    print(f"Raw response: {raw[:200]}")

    # Clean and parse
    try:
        clean = raw.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        result = json.loads(clean.strip())
        result['client_id'] = client['client_id']
        result['client_name'] = client['client_name']
        return result
    except Exception as e:
        print(f"Parse error: {e}")
        print(f"Full raw response: {raw}")
        return None


In [ ]:
def run_platform():
    print("\n👋 Welcome to the AI Banking Platform")

    while True:
        choice = show_menu()

        if choice == "1":
            print("\n🏦 CUSTOMER RISK ASSESSMENT")
            print("-" * 40)
            try:
                name = input("Customer name: ")
                balance = float(input("Balance (€): "))
                transactions = int(input("Transactions: "))
                active = input("Active? (yes/no): ").lower() == "yes"

                customer = {
                    "name": name,
                    "balance": balance,
                    "transactions": transactions,
                    "is_active": active
                }

                print("\n⏳ Analysing...")
                result = analyse_customer_structured(client_groq, customer)
                print(f"\n📋 {name.upper()}")
                print(f"Risk Score:     {result['risk_score']}/10")
                print(f"Risk Level:     {result['risk_level']}")
                print(f"Reason:         {result['reason']}")
                print(f"Recommendation: {result['recommendation']}")

            except ValueError:
                print("\n❌ Invalid input — balance and transactions must be numbers")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "2":
            print("\n📋 USER STORY GENERATOR")
            print("-" * 40)
            try:
                requirement = input("Enter business requirement: ")

                print("\n⏳ Generating...")
                story = generate_user_story(requirement)
                print(f"\n✅ USER STORY:")
                print(f"{story['user_story']}")
                print(f"\nPriority:     {story['priority']}")
                print(f"Story Points: {story['story_points']}")
                print(f"\nACCEPTANCE CRITERIA:")
                for ac in story['acceptance_criteria']:
                    print(f"  → {ac}")
                print(f"\nDEFINITION OF DONE:")
                for dod in story['definition_of_done']:
                    print(f"  ☐ {dod}")

            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "3":
            print("\n⚖️  EU AI ACT CHECKER")
            print("-" * 40)
            try:
                use_case = input("Describe your AI use case: ")

                print("\n⏳ Assessing...")
                result = check_eu_ai_act(use_case)
                print(f"\n{result['risk_tier']}")
                print(f"Reason: {result['risk_tier_reason']}")
                print(f"\nCOMPLIANCE REQUIREMENTS:")
                for req in result['compliance_requirements']:
                    print(f"  → {req}")
                print(f"\nRED FLAGS:")
                for flag in result['red_flags']:
                    print(f"  ⚠️  {flag}")
                print(f"\nRECOMMENDED ACTIONS:")
                for action in result['recommended_actions']:
                    print(f"  → {action}")

            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "4":
            print("\n🌱 EU TAXONOMY ASSESSMENT")
            print("-" * 40)
            try:
                client_id = input("Client ID: ")
                client_name = input("Client name: ")
                sector = input("Sector: ")
                nace_code = input("NACE code: ")
                main_purpose = input("Main purpose: ")
                sub_purpose = input("Sub purpose (or Enter to skip): ")
                loan_amount = float(input("Loan amount (€): "))
                ghg = input("GHG Scope 1+2 tonnes (or Enter to skip): ")

                taxonomy_client = {
                    "client_id": client_id,
                    "client_name": client_name,
                    "sector": sector,
                    "nace_code": nace_code,
                    "main_purpose": main_purpose,
                    "sub_purpose": sub_purpose or None,
                    "product_type": "Term Loan",
                    "country": "Netherlands",
                    "annual_turnover_eur": 0,
                    "capex_eur": 0,
                    "loan_amount_eur": loan_amount,
                    "reporting_year": 2024,
                    "substantial_contribution_objective": "Climate Change Mitigation",
                    "revenue_eligible_pct": None,
                    "capex_eligible_pct": None,
                    "ghg_scope1_2_tonnes": float(ghg) if ghg else None,
                    "minimum_safeguards_compliant": True
                }

                print("\n⏳ Assessing...")
                result = assess_taxonomy(taxonomy_client)
                print(f"\n🌱 {client_name.upper()}")
                print(f"Eligible:   {result['taxonomy_eligible']}")
                print(f"Aligned:    {result['taxonomy_aligned']}")
                print(f"Tier:       {result['assessment_tier']}")
                if result.get('dnsh_assessment'):
                    print(f"DNSH:       {result['dnsh_assessment']['overall']}")
                print(f"Safeguards: {result['minimum_safeguards']}")
                if result.get('gaps'):
                    print(f"\nGAPS:")
                    for gap in result['gaps']:
                        print(f"  ⚠️  {gap}")
                if result.get('recommendations'):
                    print(f"\nRECOMMENDATIONS:")
                    for rec in result['recommendations']:
                        print(f"  → {rec}")

            except ValueError:
                print("\n❌ Invalid input — loan amount must be a number")
            except Exception as e:
                print(f"\n❌ Something went wrong: {e}")

        elif choice == "0":
            print("\n👋 Goodbye!")
            break

        else:
            print("\n❌ Invalid choice. Try again.")

# Run the platform
run_platform()


👋 Welcome to the AI Banking Platform

🏦 AI BANKING PLATFORM
1. Customer Risk Assessment
2. Generate User Stories
3. EU AI Act Checker
4. EU Taxonomy Assessment
0. Exit
Choose a tool (0-4): 1

🏦 CUSTOMER RISK ASSESSMENT
----------------------------------------
Customer name: aa
Balance (€): 11111
Transactions: 1
Active? (yes/no): yes

⏳ Analysing...

📋 AA
Risk Score:     2/10
Risk Level:     Low
Reason:         The customer has a moderate balance and only one transaction, indicating a low risk of fraudulent activity.
Recommendation: Continue to monitor the customer's account activity, but no immediate action is required.

🏦 AI BANKING PLATFORM
1. Customer Risk Assessment
2. Generate User Stories
3. EU AI Act Checker
4. EU Taxonomy Assessment
0. Exit
Choose a tool (0-4): 4

🌱 EU TAXONOMY ASSESSMENT
----------------------------------------
Client ID: 1
Client name: a
Sector: a
NACE code: 1
Main purpose: a
Sub purpose (or Enter to skip): 
Loan amount (€): 1
GHG Scope 1+2 tonnes (or Enter 